### Setting

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import unicodedata
import re

from pathlib import Path
# BASE_DIR = Path().resolve().parent   
# DATA_DIR = BASE_DIR / "data"
# DATA_DIR_In = DATA_DIR / "interim"

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path("/content/drive/MyDrive/NLP_project")

def show_full(df):
    """
    Fully display a DataFrame without any truncation.
    Works for any df[...] slice.
    """
    import pandas as pd
    from IPython.display import display

    with pd.option_context(
        "display.max_colwidth", None,
        "display.max_rows", None,
        "display.max_columns", None,
        "display.width", None
    ):
        display(df)

In [2]:
import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using {device} as device.")

Using cpu as device.


### basic sanity checks

In [ ]:
text_path = DATA_DIR /"df_clean.parquet"

df = pd.read_parquet(text_path).reset_index(drop=True)
df.head(2)

,url,date,language,title,text,text_clean,doc_id
0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,en,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",0
1,https://the-decoder.com/a-changing-internet-wi...,2025-10-20,en,"""A changing internet"": Wikipedia sees drop in ...","\n""A changing internet"": Wikipedia sees drop i...","""A changing internet"": Wikipedia sees drop in ...",1


In [8]:
df['text_clean_len'] = df['text_clean'].str.len()
df.columns

Index(['url', 'date', 'language', 'title', 'text', 'text_clean', 'doc_id',
       'text_clean_len'],
      dtype='object')

In [9]:
# --- basic sanity checks ---
s = df["text_clean"].fillna("").astype(str)
t = df["title"].fillna("").astype(str)

print("N:", len(s))
print("Empty ratio:", (s.str.strip().eq("")).mean())
print("Duplicate text ratio:", (s.duplicated()).mean())
print("Duplicate title ratio:", (t.duplicated()).mean())
print("Length describe:\n", df['text_clean_len'].describe(percentiles=[.01,.05,.1,.85,.9,.95,.99]))

N: 148900
Empty ratio: 0.0
Duplicate text ratio: 0.0
Duplicate title ratio: 0.0
Length describe:
 count    148900.000000
mean       7222.464513
std        5699.102848
min         800.000000
1%         1116.000000
5%         2033.000000
10%        2820.000000
50%        6053.000000
85%       10644.150000
90%       12206.000000
95%       15227.050000
99%       27714.080000
max      227107.000000
Name: text_clean_len, dtype: float64


### Chunk long documents into paragraph/section blocks
- What I care about: industry/company/technology/impact mechanism  
- This information is usually expressed clearly in local paragraphs, no need to understand the entire article  
- Block lengths are more balanced, topics are more stable, and low-information paragraphs can be removed  
- With doc_id I can do backward aggregation.

> TARGET = 4000–6000 chars

> MIN = 800–1200 chars

> OVERLAP = 200–400 chars

In [10]:
keep_cols = ["doc_id","url", "date", "title", "text_clean",'text_clean_len']
df_blk = df[keep_cols].copy()

#### Step 1: paragraph splitting using `\n\n`

Get paragraph list p1, p2, ...

Then do packing and merging:

- Accumulate paragraphs into a block from the beginning

- Once adding the next paragraph would exceed TARGET, seal the current block and start a new one

- If a single paragraph alone is > TARGET: go to Step 2

In [11]:
import pandas as pd

TARGET_LO = 4000   # seal block once it reaches this length
TARGET_HI = 6000   # hard cap / oversize threshold
MIN_LEN   = 800    # minimum acceptable block length

def split_paragraphs(text: str):
    """Split by paragraph boundary '\\n\\n' and keep non-empty paragraphs."""
    t = "" if text is None else str(text)
    paras = [p.strip() for p in t.split("\n\n")]
    return [p for p in paras if p]

def pack_paragraphs_step1(paras, target_lo=TARGET_LO, target_hi=TARGET_HI, min_len=MIN_LEN, sep="\n\n"):
    """
    Pack paragraphs into blocks using a string container (cur_block).

    Rules (per your spec):
      1) Iterate paragraphs p in order, concatenate into cur_block (with sep if needed).
      2) If a single paragraph length > target_hi:
         - seal current cur_block (if non-empty)
         - push this paragraph as its own block (oversize)
      3) If cur_block_len >= target_lo after adding p:
         - seal cur_block into blocks and reset container
      4) After loop:
         - if last cur_block_len < min_len, append it to previous block (if exists)
           else push it as its own block.
    """
    blocks = []
    cur_block = ""

    for p in paras:
        p_len = len(p)

        # Case: single paragraph is oversize -> becomes its own block
        if p_len > target_hi:
            if cur_block:
                blocks.append(cur_block)
                cur_block = ""
            blocks.append(p)
            continue

        # Add paragraph to container
        if not cur_block:
            cur_block = p
        else:
            cur_block = cur_block + sep + p

        # Seal when informative enough
        if len(cur_block) >= target_lo:
            blocks.append(cur_block)
            cur_block = ""

    # Handle tail container
    if cur_block:
        if len(cur_block) < min_len and len(blocks) > 0:
            blocks[-1] = blocks[-1] + sep + cur_block
        else:
            blocks.append(cur_block)

    return blocks

def build_blocks_step1(df_in, text_col="text_clean", target_lo=TARGET_LO, target_hi=TARGET_HI, min_len=MIN_LEN):
    """
    Build block-level dataframe:
      each row = one block (doc_id + block_id).
    Also mark:
      is_oversize_para = 1 if len(block_text) > target_hi else 0
    """
    rows = []
    for _, r in df_in.iterrows():
        paras = split_paragraphs(r[text_col])
        blocks = pack_paragraphs_step1(paras, target_lo=target_lo, target_hi=target_hi, min_len=min_len)

        for bi, btxt in enumerate(blocks):
            rows.append({
                "doc_id": r["doc_id"],
                "block_id": bi,
                "block_char_len": len(btxt),
                "is_oversize_para": 1 if len(btxt) > target_hi else 0,
                "is_short_before_long": 1 if len(btxt) < min_len else 0,
                "block_text": btxt,
                "url": r["url"],
                "date": r["date"],
                "title": r["title"],
                "source_text_len": r['text_clean_len'],
            })

    return pd.DataFrame(rows)

In [12]:
df_blocks_step1 = build_blocks_step1(df_blk, text_col="text_clean")

In [13]:
df_blocks_step1["is_oversize_para"].value_counts()

is_oversize_para
0    205962
1     41455
Name: count, dtype: int64

In [14]:
print(
    "Block Length describe in Step 1:\n",
    df_blocks_step1.query("is_oversize_para == 0 and is_short_before_long == 0")["block_char_len"]
    .describe(percentiles=[.01,.05,.1,.85,.9,.95,.99])
)

Block Length describe in Step 1:
 count    197153.000000
mean       3540.672315
std        1340.857470
min         800.000000
1%          847.000000
5%         1047.000000
10%        1367.000000
50%        4029.000000
85%        4790.000000
90%        5066.000000
95%        5461.000000
99%        5882.000000
max        6000.000000
Name: block_char_len, dtype: float64


#### Step 2: Paragraphs still too long → Split by `\n` lines 

For paragraphs that are still too long, further split them into line/sentence-like fragments using `\n`, then perform the same packing process.

In [15]:
def split_lines(text: str):
    """
    Split by single newline '\n' and keep non-empty lines.
    """
    t = "" if text is None else str(text)
    lines = [ln.strip() for ln in t.split("\n")]
    return [ln for ln in lines if ln]

def pack_lines_step2(lines, target_lo=4000, target_hi=6000, min_len=800, sep="\n"):
    """
    Pack lines into blocks using a string container (cur_block) (same logic as Step1).

    Rules:
      1) Iterate lines in order, concatenate into cur_block (with sep if needed).
      2) If a single line length > target_hi:
         - seal current cur_block (if non-empty)
         - push this line as its own block (still oversize; defer to Step3)
      3) If cur_block_len >= target_lo after adding line:
         - seal cur_block and reset
      4) After loop:
         - if last cur_block_len < min_len, append it to previous block (if exists)
           else push it as its own block.
    """
    blocks = []
    cur_block = ""

    for ln in lines:
        ln_len = len(ln)

        # single line oversize -> own block (Step3 will handle)
        if ln_len > target_hi:
            if cur_block:
                blocks.append(cur_block)
                cur_block = ""
            blocks.append(ln)
            continue

        # add line to container
        if not cur_block:
            cur_block = ln
        else:
            cur_block = cur_block + sep + ln

        # seal when informative enough
        if len(cur_block) >= target_lo:
            blocks.append(cur_block)
            cur_block = ""

    # handle tail
    if cur_block:
        if len(cur_block) < min_len and len(blocks) > 0:
            blocks[-1] = blocks[-1] + sep + cur_block
        else:
            blocks.append(cur_block)

    return blocks

def build_blocks_step2(df_blocks_step1, target_lo=4000, target_hi=6000, min_len=800, text_col="block_text"):
    """
    Expand Step1 oversize blocks into Step2 blocks (line-level packing).
    """
    rows = []

    for _, r in df_blocks_step1.iterrows():
        if int(r["is_oversize_para"]) != 1:
            if r["is_short_before_long"] == 1:
                continue # drop
            else:
                rows.append({
                    "doc_id": r["doc_id"],
                    "block_id": r["block_id"],      # Step1 id
                    "block_id2": 0,                 # Step2 id
                    "block_char_len": len(r['block_text']),
                    "is_oversize_line":  0,         # for Step3
                    "is_short_before_long": 0,
                    "block_text": r[text_col],
                    "url": r["url"],
                    "date": r["date"],
                    "title": r["title"],
                    "source_text_len": r.get("source_text_len", None),
                })
            
        else:
            text = r[text_col]
            lines = split_lines(text)
            blocks2 = pack_lines_step2(lines, target_lo=target_lo, target_hi=target_hi, min_len=min_len)

            for bi2, b2 in enumerate(blocks2):
                b2_len = len(b2)
                source_blx_len = r['block_char_len']
                rows.append({
                    "doc_id": r["doc_id"],
                    "block_id": r["block_id"],      # Step1 id
                    "block_id2": bi2,               # Step2 id
                    "block_char_len": b2_len,
                    "is_oversize_line": 1 if b2_len > target_hi else 0,  # for Step3
                    "is_short_before_long": 1 if b2_len < min_len else 0,
                    "block_text": b2,
                    "url": r["url"],
                    "date": r["date"],
                    "title": r["title"],
                    "source_text_len": str(r.get("source_text_len", ""))+"_"+ str(source_blx_len),
                })

    return pd.DataFrame(rows)

In [16]:
df_blocks_step2 = build_blocks_step2(df_blocks_step1, target_lo=4000, target_hi=6000, min_len=800)

In [17]:
df_blocks_step2.head()

,doc_id,block_id,block_id2,block_char_len,is_oversize_line,is_short_before_long,block_text,url,date,title,source_text_len
0,0,0,0,9294,1,0,"""A Gift To Humanity"": AlphaFold AI Predicts St...",https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",9294_9294
1,1,0,0,2615,0,0,"""A changing internet"": Wikipedia sees drop in ...",https://the-decoder.com/a-changing-internet-wi...,2025-10-20,"""A changing internet"": Wikipedia sees drop in ...",2616
2,2,0,0,2201,0,0,"""A first for the city"": Four Naples intersecti...",https://www.fox4now.com/naples/a-first-for-the...,2024-04-18,"""A first for the city"": Four Naples intersecti...",2203
3,3,0,0,5064,0,0,"""AI Ecosystem Cultivation"": Cultivating A Stro...",https://celebritiesdeaths.com/ai-ecosystem-cul...,2023-06-02,"""AI Ecosystem Cultivation"": Cultivating A Stro...",5064
4,4,0,0,2281,0,0,"""AI Fever!"" AI Is Animated Idiocy! Can This Hu...",https://thetaoofanarchy.substack.com/p/ai-feve...,2025-01-30,"""AI Fever!"" AI Is Animated Idiocy! Can This Hu...",2281


In [18]:
print("Rows step1:", len(df_blocks_step1))
print("Rows step2:", len(df_blocks_step2))  

print("Pct <800:", len(df_blocks_step2[df_blocks_step2["block_char_len"] < 800]))

Rows step1: 247417
Rows step2: 261888
Pct <800: 2851


In [19]:
df_blocks_step2["is_oversize_line"].value_counts()

is_oversize_line
0    233672
1     28216
Name: count, dtype: int64

In [20]:
print(
    "Block Length describe in Step 2:\n",
    df_blocks_step2.query("is_oversize_line == 0 and is_short_before_long == 0")["block_char_len"]
    .describe(percentiles=[.01,.05,.1,.85,.9,.95,.99])
)

Block Length describe in Step 2:
 count    230821.000000
mean       3524.816520
std        1321.661671
min         800.000000
1%          848.000000
5%         1056.000000
10%        1397.000000
50%        4028.000000
85%        4738.000000
90%        5013.000000
95%        5425.000000
99%        5874.000000
max        6000.000000
Name: block_char_len, dtype: float64


#### Step 3: Still no separator (entire text is one line) → Sentence splitting 

In [21]:
SENT_SPLIT_SIMPLE = re.compile(r'(?<=[.!?])\s+(?=[A-Z0-9"])')

# 1) protect abbreviations / acronyms / decimals
ABBR_PATTERNS = [
    r'\b(?:Mr|Mrs|Ms|Dr|Prof|Inc|Ltd|Co|Corp|Jr|Sr|St|No)\.',   # common abbrevs
    r'\b(?:[A-Z]\.){2,}',                                       # U.S. / U.K. / E.U.
    r'\b\d+\.\d+\b',                                            # decimals 3.14
]

def protect_dots(text: str) -> str:
    t = "" if text is None else str(text)
    # protect abbrev dots: "Dr." -> "Dr§"
    for p in ABBR_PATTERNS:
        t = re.sub(p, lambda m: m.group(0).replace(".", "§"), t)
    return t

def unprotect_dots(text: str) -> str:
    return text.replace("§", ".")

def split_sentences_regex(text: str):
    t = "" if text is None else str(text).strip()
    if not t:
        return []
    t2 = protect_dots(t)
    parts = [s.strip() for s in SENT_SPLIT_SIMPLE.split(t2) if s.strip()]
    return [unprotect_dots(s) for s in parts]

def pack_sentences_step3(sentences, target_lo=4000, target_hi=6000, min_len=800):
    """
    Pack sentence list into blocks using container accumulation logic.
    """
    blocks = []
    cur_block = ""

    for sent in sentences:
        sent_len = len(sent)

        if sent_len > target_hi:
            if cur_block:
                blocks.append(cur_block)
                cur_block = ""
            blocks.append(sent)
            continue

        if not cur_block:
            cur_block = sent
        else:
            cur_block = cur_block + " " + sent

        if len(cur_block) >= target_lo:
            blocks.append(cur_block)
            cur_block = ""

    if cur_block:
        if len(cur_block) < min_len and len(blocks) > 0:
            blocks[-1] = blocks[-1] + " " + cur_block
        else:
            blocks.append(cur_block)

    return blocks

def build_blocks_step3(df_blocks_step2,
                       target_lo=4000,
                       target_hi=6000,
                       min_len=800,
                       text_col="block_text"):

    rows = []

    for _, r in df_blocks_step2.iterrows():

        # non oversize_line
        if int(r["is_oversize_line"]) != 1:
            if r["is_short_before_long"] == 1:
                continue # drop
            else:
                rows.append({
                    "doc_id": r["doc_id"],
                    "block_id": r["block_id"],
                    "block_id2": r["block_id2"],
                    "block_id3": 0,
                    "block_char_len": r["block_char_len"],
                    "is_still_oversize": 0,
                    "block_text": r[text_col],
                    "url": r["url"],
                    "date": r["date"],
                    "title": r["title"],
                    "source_text_len": r.get("source_text_len", None),
                })

        else:
            # ---- oversize_line -> sentence split ----
            text = r[text_col]
            sentences = split_sentences_regex(text)
            blocks3 = pack_sentences_step3(
                sentences,
                target_lo=target_lo,
                target_hi=target_hi,
                min_len=min_len
            )

            for bi3, b3 in enumerate(blocks3):
                b3_len = len(b3)
                source_blx_len = r['block_char_len']
                rows.append({
                    "doc_id": r["doc_id"],
                    "block_id": r["block_id"],
                    "block_id2": r["block_id2"],
                    "block_id3": bi3,
                    "block_char_len": b3_len,
                    "is_still_oversize": 1 if b3_len > target_hi else 0,
                    "block_text": b3,
                    "url": r["url"],
                    "date": r["date"],
                    "title": r["title"],
                    "source_text_len":str(r.get("source_text_len", ""))+"_"+ str(source_blx_len),
                })

    return pd.DataFrame(rows)

In [22]:
df_blocks_step3 = build_blocks_step3(df_blocks_step2)

In [23]:
print("Pct still oversize:", df_blocks_step3["is_still_oversize"].mean())
print("Min len:", df_blocks_step3["block_char_len"].min())
print(df_blocks_step3["block_char_len"].describe(percentiles=[.01,.05,.5,.9,.95,.99]))

Pct still oversize: 0.015364951227716112
Min len: 4
count    297300.000000
mean       3605.730888
std        1362.111153
min           4.000000
1%          851.000000
5%         1100.000000
50%        4042.000000
90%        5040.000000
95%        5527.000000
99%        6510.010000
max       29008.000000
Name: block_char_len, dtype: float64


In [24]:
check = df_blocks_step3.query("block_char_len < 800 or block_char_len > 6000")
print(len(check))

4833


In [ ]:
# 1) keep only blocks in [800, 6000]
df_blocks_ = df_blocks_step3[
    df_blocks_step3["block_char_len"].between(800, 6000)
].copy()

# 2) build blk_id = "<block_id>_<block_id2>_<block_id3>"
df_blocks_["blk_id"] = (
    df_blocks_["block_id"].astype(str) + "_" +
    df_blocks_["block_id2"].astype(str) + "_" +
    df_blocks_["block_id3"].astype(str)
)

# 3) select final columns
df_blocks = df_blocks_[["doc_id", "blk_id", "url", "date", "title", "block_text", "block_char_len"]].copy()

print(len(df_blocks))
print(df_blocks["block_char_len"].describe(percentiles=[.01,.05,.5,.9,.95,.99]))

df_blocks.head(2)

In [ ]:
df_blocks_path = DATA_DIR / "df_blocks.parquet"
df_blocks.to_parquet(df_blocks_path, index=False)

### Bertopic

In [48]:
def topic_overview(topic_ids, topn_words=20, df_blk_topics=df_blk_topics, topic_model=topic_model):
    topic_set = set(topic_ids)
    counts = (df_blk_topics[df_blk_topics["topic"].isin(topic_set)]
              .groupby("topic").size().rename("n_docs"))

    rows = []
    for tid, n_docs in counts.items():
        kw = topic_model.get_topic(int(tid)) or []
        rep = ", ".join([w for w, _ in kw[:topn_words]])
        rows.append({"topic": int(tid), "representation": rep, "n_docs": int(n_docs)})
    
    df_overview = pd.DataFrame(rows).sort_values("n_docs", ascending=False).reset_index(drop=True)

    print("# of topics in the pool:", len(df_overview))
    print("# of texts in these topics:", df_overview["n_docs"].sum())
    return df_overview

def inspect_topic(
    tid,
    n=5,
    mode="head",                # "head" or "random"
    text_col="block_text",
    show_chars=3000,
    random_state=42,
    prob_th=0.5,
    check_pool=None             # list / set of topic ids
):
    # ---- pool filter ----
    if check_pool is not None and tid not in set(check_pool):
        return print(f"Topic {tid} not in check_pool.")

    print("=" * 100)
    print(f"TOPIC: {tid}")

    # ---- keywords ----
    kw = topic_model.get_topic(tid) or []
    kw20 = [(w, round(s, 4)) for w, s in kw[:20]]
    print("Top20 Keywords:", kw20)

    if "AI_RE" in globals():
        hit = bool(AI_RE.search(" ".join([w for w, _ in kw20])))
        print("AI_hit_in_kw20:", hit)

    print("=" * 100)

    # ---- all docs in this topic ----
    sub_all = df_blk_topics[df_blk_topics["topic"] == tid]
    print("Total docs in topic:", len(sub_all))

    # ---- apply probability filter ----
    sub = sub_all[sub_all["probability"] >= prob_th].copy()

    if sub.empty:
        return print("No docs after prob_th filter.")

    sub = sub.sort_values("probability", ascending=False)

    print("Docs (prob >=", prob_th, "):", len(sub),
          "| mean_prob:", round(sub["probability"].mean(), 3))
    print()

    # ---- sampling mode ----
    if mode == "random":
        sub_view = sub.sample(min(n, len(sub)), random_state=random_state)
    else:
        sub_view = sub.head(n)

    # ---- print docs ----
    for i, r in enumerate(sub_view.itertuples(), 1):
        txt = str(getattr(r, text_col, ""))
        print(f"[{i}] prob={r.probability:.3f} | len={len(txt)} | "
              f"{str(getattr(r,'title',''))[:120]}")
        print("-" * 80)
        print(txt[:show_chars])
        print()

#### Try 1: Filter those topic are typical boilerplate

In [16]:
df_blocks_path = DATA_DIR / "df_blocks.parquet"
df_blocks = pd.read_parquet(df_blocks_path)
df_blocks.head(2)

,doc_id,blk_id,url,date,title,block_text,block_char_len
0,0,0_0_0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",4123
1,0,0_0_1,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",Advertisement“It is allowing us to predict pro...,5170


In [17]:
print(len(df_blocks))
print(df_blocks["block_char_len"].describe(percentiles=[.01,.05,.5,.9,.95,.99]))

292467
count    292467.000000
mean       3552.132900
std        1276.399934
min         800.000000
1%          856.000000
5%         1101.000000
50%        4038.000000
90%        4940.000000
95%        5359.000000
99%        5856.000000
max        6000.000000
Name: block_char_len, dtype: float64


In [8]:
### customize bertopic model
# ! pip install bertopic
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer

RANDOM_STATE = 27

embedding_model = SentenceTransformer("thenlper/gte-small", device="cuda") 

umap_model = UMAP(
    n_neighbors=50, # smaller n_neighbors will -> local structure -> more detailed topics
    n_components=15, # smaller n_components will -> more coarse topics
    min_dist=0.2, # smaller min_dist will -> more clustered紧凑 topics
    metric="cosine",
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=120, # smaller min_cluster_size will -> more detailed topics
    min_samples=30, # smaller min_samples -> easier get into topics -> more detailed topics, but also more noise
    metric="euclidean",
    cluster_selection_method="eom"
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=20,
    # max_df=0.95 # defualt 1.0
)
ctfidf_model = ClassTfidfTransformer()

# All steps together
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# df_small = df_tm.sample(20000, random_state=RANDOM_STATE)
docs = df_blocks["block_text"].tolist()
topics, probs = topic_model.fit_transform(docs)

In [ ]:
df_blk_topics1 = df_blocks.reset_index(drop=True).assign(topic=topics, probability=probs)

print("N blocks:", len(df_blk_topics1))
print("N topics (excl -1):", df_blk_topics1.query("topic!=-1")["topic"].nunique())
print("Noise ratio (-1):", (df_blk_topics1["topic"]==-1).mean())
print("Mean prob (non-noise):", df_blk_topics1.query("topic!=-1")["probability"].mean())

N blocks: 292467
N topics (excl -1): 291
Noise ratio (-1): 0.495638824209227
Mean prob (non-noise): 0.8267431581289714


In [21]:
with pd.option_context('display.max_rows', None):
    display(topic_model.get_topic_info())

,Topic,Count,Name,Representation,Representative_Docs
0,-1,144958,-1_said_chatgpt_google_generative,"[said, chatgpt, google, generative, models, op...","[The startup’s Agent, through its Agent Operat..."
1,0,8833,0_share price_price_india_vs,"[share price, price, india, vs, bank, watch, b...",[You may be interested in\n\nMilestone Alert!\...
2,1,5941,1_healthcare_medical_patient_cancer,"[healthcare, medical, patient, cancer, clinica...",[Posted in: Device / Technology News | Medical...
3,2,4458,2_star_reveals_husband_dress,"[star, reveals, husband, dress, fans, daughter...",[Sean Diddy Combs asks judge to dismiss claim ...
4,3,3386,3_students_teachers_education_student,"[students, teachers, education, student, schoo...","[In some cases, the cheating is obvious, says ..."
5,4,3257,4_currencies_daily arabic_daily english_englis...,"[currencies, daily arabic, daily english, engl...",[Jordan Committed To Building An Integrated Na...
6,5,2881,5_india_indian_minister_modi,"[india, indian, minister, modi, india ai, summ...","[IndiaAI, Microsoft join hands to harness pote..."
7,6,2821,6_banking_real estate_fraud_estate,"[banking, real estate, fraud, estate, credit, ...","[As consumer awareness of AI grows, there is i..."
8,7,2671,7_republic_fcc_public file_county,"[republic, fcc, public file, county, fcc publi...","[Students, parents fill school board meeting i..."
9,8,2668,8_crypto_blockchain_presale_token,"[crypto, blockchain, presale, token, bitcoin, ...",[Telegram: https://t.me/OzakAGI\nTwitter: http...


In [5]:
topic_path1 = DATA_DIR /"df_blk_topics.parquet"
# df_blk_topics.to_parquet(topic_path1, index=False)
df_blk_topics1 = pd.read_parquet(topic_path1)

##### Inspect on first topic modeling results

In [5]:
import re

AI_TERMS = [
    r"artificial intelligence", r"\bai\b", r"\bgenerative\s+ai\b", r"\bgenai\b",
    r"\bllm\b", r"large language model[s]?",
    r"\bmachine learning\b", r"\bml\b", r"\bdeep learning\b",
    r"\bchatgpt\b", r"\bgpt-?\d*\b", r"\bopenai\b", r"\banthropic\b", r"\bclaude\b",
    r"\bgemini\b", r"\bgoogle\s+ai\b", r"\bmicrosoft\s+copilot\b", r"\bcopilot\b",
    r"\bstable diffusion\b", r"\bmidjourney\b", r"\bdall[- ]?e\b",
    r"\blangchain\b", r"\brag\b", r"retrieval-augmented generation",
    r"\btransformer(s)?\b", r"\bfoundation model(s)?\b"
]
AI_RE = re.compile("|".join(AI_TERMS), flags=re.IGNORECASE)

topic_ids = sorted(set(df_blk_topics["topic"]) - {-1})

nonai_topics = []
ai_topics = []

for tid in topic_ids:
    kw = topic_model.get_topic(tid) or []
    kw_words = " ".join([w for w, _ in kw[:20]])
    (ai_topics if AI_RE.search(kw_words) else nonai_topics).append(tid)

print("AI topics:", len(ai_topics))
print("Non-AI topics:", len(nonai_topics))

: 

In [ ]:
topic_stats = (
    df_blk_topics1
      .groupby("topic")["probability"]
      .agg(
          count="size",
          purity_mean="mean",
          purity_median="median",
          q05=lambda x: np.quantile(x, 0.05),
          q10=lambda x: np.quantile(x, 0.10),
          q20=lambda x: np.quantile(x, 0.20),
          high_conf_ratio=lambda x: np.mean(x >= 0.7)
      )
      .sort_values("purity_mean", ascending=False)
)

topic_stats["purity_label"] = np.where(topic_stats["q20"] < 0.5,"impure", "pure")
impure_topics = topic_stats.query("purity_label == 'impure'").index.tolist()
pure_topics   = topic_stats.query("purity_label == 'pure'").index.tolist()

print("Impure topics:", len(impure_topics))
print("Pure topics:", len(pure_topics))

Impure topics: 62
Pure topics: 230


In [25]:
nonai_pure_topics = sorted(set(nonai_topics).intersection(pure_topics))
print("nonai_pure_topics:", len(nonai_pure_topics))
nonai_impure_topics = sorted(set(nonai_topics).intersection(impure_topics))
print("nonai_impure_topics:", len(nonai_impure_topics))

nonai_pure_topics: 139
nonai_impure_topics: 41


In [39]:
nonai_pure_topics[:10]

[0, 1, 2, 4, 6, 7, 8, 10, 12, 13]

##### Cleaning based on Inspectation

In [6]:
topic_to_delete = [0,2,7,55,
                   112,114,131,158,164,167,172,175,178,181,186,188,196,198,
                   210,231,240,244,245,248,254,269,271,279,282,284,289,290]

df_blk_topics_clean1 = df_blk_topics1[ ~df_blk_topics1["topic"].isin(topic_to_delete)].copy()
df_blk_topics_clean1

,doc_id,blk_id,url,date,title,block_text,block_char_len,topic,probability
0,0,0_0_0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",4123,1,1.000000
1,0,0_0_1,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",Advertisement“It is allowing us to predict pro...,5170,1,1.000000
2,1,0_0_0,https://the-decoder.com/a-changing-internet-wi...,2025-10-20,"""A changing internet"": Wikipedia sees drop in ...","""A changing internet"": Wikipedia sees drop in ...",2615,228,0.862802
3,2,0_0_0,https://www.fox4now.com/naples/a-first-for-the...,2024-04-18,"""A first for the city"": Four Naples intersecti...","""A first for the city"": Four Naples intersecti...",2201,-1,0.000000
4,3,0_0_0,https://celebritiesdeaths.com/ai-ecosystem-cul...,2023-06-02,"""AI Ecosystem Cultivation"": Cultivating A Stro...","""AI Ecosystem Cultivation"": Cultivating A Stro...",5064,-1,0.000000
...,...,...,...,...,...,...,...,...,...
292462,148898,0_0_0,https://www.siliconrepublic.com/business/200bn...,2025-02-11,€200bn more mobilised for AI in Europe,€200bn more mobilised for AI in Europe\n\nThe ...,4982,-1,0.000000
292463,148898,1_0_0,https://www.siliconrepublic.com/business/200bn...,2025-02-11,€200bn more mobilised for AI in Europe,"US, France, Europe, AI, funding and investment...",1852,-1,0.000000
292464,148899,0_0_0,https://www.storyboard18.com/digital/%E2%82%B9...,2025-10-28,"₹12,000 cr on mute: OpenAI’s Generative music ...","₹12,000 cr on mute: OpenAI’s Generative music ...",1472,9,0.988283
292465,148899,0_1_0,https://www.storyboard18.com/digital/%E2%82%B9...,2025-10-28,"₹12,000 cr on mute: OpenAI’s Generative music ...","The creative advantageAkshay Mathur, founder a...",4222,9,1.000000


In [7]:
topic_path2 = DATA_DIR / "df_blk_topics_clean1.parquet"
df_blk_topics_clean1.to_parquet(topic_path2, index=False)

#### Try 2: with vectorizer used for noise reduction
跑完后，对 topic 的 top words 做两类评分：
	•	AI_score（AI 词典命中率）
	•	Boilerplate_score（模板词命中率）
然后批量 topic_to_delete / 或标记 doc 为非目标。

In [3]:
topic_path2 = DATA_DIR / "df_blk_topics_clean1.parquet"
df_blk_topics_clean1 = pd.read_parquet(topic_path2)

In [4]:
print(len(df_blk_topics_clean1))
df_blk_topics_clean1.head(2)

270518


,doc_id,blk_id,url,date,title,block_text,block_char_len,topic,probability
0,0,0_0_0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",4123,1,1.0
1,0,0_0_1,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",Advertisement“It is allowing us to predict pro...,5170,1,1.0


In [ ]:
# ! pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 14.8 MB/s eta 0:00:00


In [10]:
### customize bertopic model
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from sklearn.cluster import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

RANDOM_STATE = 27

embedding_model = SentenceTransformer("thenlper/gte-small", device="cuda") 

umap_model = UMAP(
    n_neighbors=50, # smaller n_neighbors will -> local structure -> more detailed topics
    n_components=15, # smaller n_components will -> more coarse topics
    min_dist=0.2, # smaller min_dist will -> more clustered紧凑 topics
    metric="cosine",
    random_state=RANDOM_STATE
)

hdbscan_model = HDBSCAN(
    min_cluster_size=200, # smaller min_cluster_size will -> more detailed topics
    min_samples=60, # smaller min_samples -> easier get into topics -> more detailed topics, but also more noise
    metric="euclidean",
    cluster_selection_method="eom"
)

vectorizer_model = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=8,                 # 关键：别太高，保住 gpt/openai/llm
    max_df=0.70,              # 砍掉跨站模板词，但不至于太猛
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{3,}\b",
    max_features=120_000      # 控住内存；你数据大，建议加
    # 你如果想加自定义 stopwords，见下方“可选增强”
)
ctfidf_model = ClassTfidfTransformer()

# All steps together
topic_model2 = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=umap_model,                    # Step 2 - Reduce dimensionality
  hdbscan_model=hdbscan_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# df_small = df_tm.sample(20000, random_state=RANDOM_STATE)
docs = df_blk_topics_clean1["block_text"].tolist()
topics, probs = topic_model2.fit_transform(docs)

##### 4 representations with different objectives


06：更强的全局高频词过滤（max_df=0.6）→ 更“干净”的关键词，用来对照降噪力度

07：较保守的高频词过滤（max_df=0.7）→ 作为 baseline 对照

07_ad：加入网页/新闻模板 stopwords 的 bigram 表达 → 最适合展示/解释 topic

uni：unigram + domain stopwords → 最适合做 AI relevance / 垃圾 topic 自动过滤与打分

In [12]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
import copy

DOMAIN_STOP = [
    "cookie","cookies","consent","privacy","gdpr","ccpa","terms","policy",
    "subscribe","subscription","signin","login","register","account",
    "newsletter","copyright","ads",
    "javascript","enable","disabled","browser","continue","reading",
    "share","facebook","twitter","linkedin","email","whatsapp",
    "click","read","more","rights","reserved"
]
AI_STOP = [
    "ai","artificial","intelligence","artificial-intelligence",
    "gpt","gpt-4","gpt4","chatgpt","openai","llm","llms",
    "generative","genai","copilot","transformer","diffusion"
]
custom_stop = sorted(set(ENGLISH_STOP_WORDS).union(DOMAIN_STOP))
STOP_INDUSTRY = sorted(set(custom_stop).union(AI_STOP))

vec_06 = CountVectorizer(
    stop_words=STOP_INDUSTRY,
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.60,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{3,}\b"
)
vec_07 = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=20,
    max_df=0.70,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]{2,}\b"
)
vec_07_ad = CountVectorizer(
    stop_words=custom_stop,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.70,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]{1,}\b"
)
vec_uni = CountVectorizer(
    stop_words=custom_stop,
    ngram_range=(1, 1),
    min_df=3,
    max_df=0.80,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9\-]{1,}\b"
)

In [13]:
import copy
topic_model2_06 = copy.deepcopy(topic_model2)
topic_model2_06.update_topics(docs, topics=topics, vectorizer_model=vec_06)
info_06 = topic_model2_06.get_topic_info()

topic_model2_07 = copy.deepcopy(topic_model2)
topic_model2_07.update_topics(docs, topics=topics, vectorizer_model=vec_07)
info_07 = topic_model2_07.get_topic_info()

topic_model2_07_ad = copy.deepcopy(topic_model2)
topic_model2_07_ad.update_topics(docs, topics=topics, vectorizer_model=vec_07_ad)
info_07_ad = topic_model2_07_ad.get_topic_info()

topic_model2_uni = copy.deepcopy(topic_model2)
topic_model2_uni.update_topics(docs, topics=topics, vectorizer_model=vec_uni)
info_uni = topic_model2_uni.get_topic_info()

2026-03-04 07:03:01,106 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-03-04 07:06:08,630 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-03-04 07:09:32,356 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF em

In [14]:
# ==== join 4 representation ====
topic2_summary = (
    info_06[["Topic","Count","Representation","Representative_Docs"]]
    .rename(columns={"Representation":"Rep_06"})
    .merge(
        info_07[["Topic","Representation"]]
        .rename(columns={"Representation":"Rep_07"}),
        on="Topic",
        how="left"
    )
    .merge(
        info_07_ad[["Topic","Representation"]]
        .rename(columns={"Representation":"Rep_07_ad"}),
        on="Topic",
        how="left"
    )
    .merge(
        info_uni[["Topic","Representation"]]
        .rename(columns={"Representation":"Rep_uni"}),
        on="Topic",
        how="left"
    )
)

# topic summary AND block-topic df
topic2_summary["Share"] = topic2_summary["Count"] / topic2_summary["Count"].sum()
cols = [c for c in topic2_summary.columns if c != "Representative_Docs"] + ["Representative_Docs"]
topic2_summary = topic2_summary[cols]

df_blk_topics2 = df_blk_topics_clean1.assign( topic=topics,  probability=probs)

In [15]:
topic2_summary.to_csv(f"{DATA_DIR}/topic2_summary.csv", index=False)
df_blk_topics2.to_parquet(f"{DATA_DIR}/df_blk_topics2.parquet", index=False)

In [3]:
DATA_DIR = "G:/我的云端硬盘/NLP_project"
topic2_summary = pd.read_csv(f"{DATA_DIR}/topic2_summary.csv")
df_blk_topics2 = pd.read_parquet(f"{DATA_DIR}/df_blk_topics2.parquet")

In [4]:
import ast
from collections import Counter

cols = ["Rep_06", "Rep_07", "Rep_07_ad"]

def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    return []

def split_kw(row):
    kws = sum([to_list(row[c]) for c in cols], [])
    cnt = Counter(kws)
    return pd.Series([
        [k for k, v in cnt.items() if v >= 2],
        [k for k, v in cnt.items() if v == 1]
    ])

topic2_summary[["dup_kw","notdup_kw"]] = topic2_summary.apply(split_kw, axis=1)
topic2_summary.head()

,Topic,Count,Rep_06,Rep_07,Rep_07_ad,Rep_uni,Share,Representative_Docs,dup_kw,notdup_kw
0,-1,146540,"['digi', 'digi communications', 'fool', 'motle...","['digi', 'digi communications', 'share price',...","['digi', 'digi communications', 'prnewswire', ...","['nasdaq', 'digi', 'altman', 'llms', 'prnewswi...",0.541701,"[""BBAI Stock’s Price Performance & Valuation T...","[digi, digi communications, fool, motley, prne...","[communications announces, motley fool, forwar..."
1,0,6987,"['drug discovery', 'breast', 'diseases', 'diag...","['clinical', 'imaging', 'drug discovery', 'dia...","['clinical', 'imaging', 'drug discovery', 'dia...","['clinical', 'disease', 'imaging', 'doctors', ...",0.025828,"[""Lunit to Power Germany's Largest Private Rad...","[drug discovery, breast, diseases, diagnostic,...","[physicians, oncology, patient care]"
2,1,3506,"['educators', 'classroom', 'cheating', 'assign...","['teachers', 'educators', 'classroom', 'cheati...","['teachers', 'educators', 'classroom', 'cheati...","['teachers', 'educators', 'classroom', 'teachi...",0.012960,['Schools prepare for Round 2 with AI | WYTV\n...,"[educators, classroom, cheating, assignments, ...",[classrooms]
3,2,3303,"['daily arabic', 'daily english', 'english dai...","['daily arabic', 'daily english', 'english dai...","['daily arabic', 'daily english', 'english dai...","['currencies', 'arabic', 'mena', 'menafn', 'in...",0.012210,['World and Middle East business and financial...,"[daily arabic, daily english, english daily, s...","[research weather, east business]"
4,3,2944,"['presale', 'mexc', 'solana', 'defi', 'price a...","['presale', 'xrp', 'decentralized', 'ethereum'...","['presale', 'ozak', 'xrp', 'ozak ai', 'decentr...","['presale', 'token', 'ozak', 'xrp', 'decentral...",0.010883,"['While XRP Holds $3, Ozak AI at $0.005 Offers...","[presale, mexc, solana, xrp, decentralized, et...","[defi, price analysis, staking, pepe, on-chain..."


In [26]:
show_full(topic2_summary[["Topic",'dup_kw']])

,Topic,dup_kw
0,-1,"[digi, digi communications, fool, motley, prnewswire, agentic, traded, workflows, gpus]"
1,0,"[drug discovery, breast, diseases, diagnostic, clinicians, diagnostics, protein, clinical, imaging, diagnosis]"
2,1,"[educators, classroom, cheating, assignments, faculty, essay, essays, professors, higher education, teachers]"
3,2,"[daily arabic, daily english, english daily, stocks currencies, europe arab, asia africa, world middle, mena, arabic, oil energy]"
4,3,"[presale, mexc, solana, xrp, decentralized, ethereum, tokens, meme]"
5,4,"[financial institutions, lenders, fraud detection, zest, credit union, credit unions, lending, mortgage, advisors]"
6,5,"[plugins, voice mode, gpts, gpt-5, gpt-4o, chatgpt plus, gpt-3, use chatgpt, chatgpt app, chat gpt]"
7,6,"[pradesh, telangana, indiaai, narendra, andhra, iit, hyderabad, crore, tcs]"
8,7,"[music industry, album, musicians, drake, grammy, musical, lyrics, suno, music group]"
9,8,"[ghana, nigerian, continent, kenya, south african, lagos, tinubu, africans, nigeria]"


### Mapping to Industries

The dissemination mechanism of AI impact in industries is typically divided into three categories:

1. Technology / Compute-driven and digital technology


2. Knowledge Content / Cognitive-driven


3. Physical / Operational-driven）

Since the project object is to Identify industries and their companies that are most likely to be impacted by AI over the next several years, we will ignore those topics that are not related to a specific industry.

In [20]:
import openai
from openai import OpenAI
DS_API_KEY = "sk-a0a8781acaa74c52acd7e22c4249185c"
client = OpenAI(
    api_key=DS_API_KEY,
    base_url="https://api.deepseek.com",
)


In [21]:
import json
import pandas as pd
from jsonschema import validate, ValidationError

import openai
from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

# =========================================================
# 1. System prompt
# =========================================================
system_msg = """
You are an expert Industry Analyst and Data Scientist.

Your task is to classify a BERTopic topic into EXACTLY ONE of the following 10 predefined main industry categories
based primarily on its keyword representations, while using Representative_Docs only as a review mechanism.

### The 10 Main Industry Categories
1. "Physical Production"
   (includes but is not limited to Agriculture, Mining, Energy, Non-digital Products Manufacturing, and Digital Products Manufacturing)

2. "Infrastructure Systems"
   (includes but is not limited to Utilities, Construction, Transportation, and Smart Cities)

3. "Consumer Economy"
   (includes but is not limited to Wholesale & Retail, E-Commerce, Food Services, and Accommodation)

4. "Financial Systems"
   (includes but is not limited to Capital Markets, Insurance, FinTech, Digital Currency, Real Estate, and Rental & Leasing)

5. "Healthcare Systems"
   (includes but is not limited to Health Services, Health Technology, and Medical Devices)

6. "Information & Digital Services"
   (the main value output is information processing, software, or digital services; includes but is not limited to Software, Cloud, IT Services, Data Platforms, AI Services, Internet Platforms, and Cybersecurity)

7. "Media & Creative Industries"
   (content as a product; includes but is not limited to News & Publishing, Arts, Film, Music, Gaming, and Social Media Content)

8. "Education & Research"
   (knowledge as a product; includes but is not limited to Education, Training, Academic Research, and Research Services)

9. "Public Sector & Civil Society"
   (includes but is not limited to Government, Public Services, NGOs, Foundations, and Civic Organizations)

10. "Other"

### Classification Rules
- Choose exactly ONE industry category.
- Base the classification primarily on the semantic meaning of the keywords.
- Frequent/repeated keywords should be weighted more heavily than infrequent keywords.
- Prefer one of the first 9 categories only when there is clear, domain-specific industry evidence.

### Definition: Boilerplate or Web Crawl Artifacts
Boilerplate or Web Crawl Artifacts refer to non-substantive webpage content
that does NOT represent the main topical content of the article.
This includes but is not limited to:

- navigation text or menu structures
- cookie/privacy notices
- subscription prompts or newsletter sign-ups
- duplicated titles or headers
- author names, dates, metadata
- social sharing buttons or related article links
- generic calls to action
- repeated layout fragments
- template-driven webpage elements
- truncated or fragmented crawl residue
- unrelated mixed webpage segments
- advertisement blocks or sponsored content modules
- concatenated button labels or short UI phrases (e.g., “WATCH”, “Live”, “Advertise With Us”, “Submit Events”, etc.)
- long sequences of short capitalized words or UI items strung together without sentence structure
- channel lists, program schedules, weather modules, or multi-section navigation directories
- footer content including copyright notices, network information, legal disclaimers
- repeated site-wide boilerplate text appearing across multiple articles
- content blocks that appear to be auto-inserted promotional, navigation, or platform framework elements

### Strict Rules for Representative_Docs
- Representative_Docs are only a review mechanism, not the primary basis for classification.
- Treat Representative_Docs conservatively.
- Do NOT treat Boilerplate or Web Crawl Artifacts as valid topical evidence.
- If industry-related words appear only within Boilerplate or Web Crawl Artifacts, ignore them.
- Only use Representative_Docs as support if they contain substantive, topic-specific content.

### Evidence Override Rule (80% Boilerplate Rule)
- If Representative_Docs appear to consist of more than 80% Boilerplate or Web Crawl Artifacts,
  AND the extracted keywords are primarily derived from such non-substantive content,
  you MUST disregard the keyword-based industry signal and classify the topic as "Other".

### Conservative Decision Rule
- Only classify into one of the first 9 industries if there is clear, domain-specific evidence in either:
    (a) the keywords from substantive content, or
    (b) substantive, topic-specific representative text.
- If both the keywords and representative text are weak, generic, mixed, noisy,
  or Boilerplate-dominated, you MUST classify the topic as "Other".

In that case:
  - "industry": "Other"
  - "relevant_keywords": []
  - "reasoning": "No obvious industry information." or "Mixed industry information."
  - "reviewing": "mostly boilerplate" or "unclear"

### Output Requirements
- Return valid JSON only.
- The "industry" field must be EXACTLY one of the 10 category names above.
- The "relevant_keywords" field must include only the most informative input keywords supporting the decision.
- The "reasoning" field must briefly explain the semantic connection between the keywords and the chosen industry.
- The "reviewing" field must:
  1. briefly summarize the main topic of Representative_Docs in a few words or a short phrase, and
  2. state whether the text is aligned with the chosen industry, mixed, or mostly boilerplate/uninformative.
- If no clear industry signal exists, return:
  - "industry": "Other"
  - "relevant_keywords": []
  - "reasoning": "No obvious industry information."
  - "reviewing": "No clear topic in representative text or mostly boilerplate."
"""

# =========================================================
# 2. User prompt template
# =========================================================
topic_classification_user_template = """
Here are the keyword representations for the topic.

Frequent keywords (higher importance; appeared >= 2 times):
{dup_kw}

Infrequent keywords (supporting evidence; appeared 1 time):
{notdup_kw}

Representative text as reference:
{Representative_Docs}

Please weigh the frequent keywords more heavily, identify the single best industry category,
review whether the representative text supports that choice, and return JSON only.
"""

# =========================================================
# 3. JSON schema
# =========================================================
topic_classification_schema = {
    "type": "object",
    "properties": {
        "industry": {
            "type": "string",
            "enum": [
                "Physical Production",
                "Infrastructure Systems",
                "Consumer Economy",
                "Financial Systems",
                "Healthcare Systems",
                "Information & Digital Services",
                "Media & Creative Industries",
                "Education & Research",
                "Public Sector & Civil Society",
                "Other"
            ]
        },
        "relevant_keywords": {
            "type": "array",
            "items": {"type": "string"}
        },
        "reasoning": {
            "type": "string"
        },
        "reviewing": {
            "type": "string"
        }
    },
    "required": ["industry", "relevant_keywords", "reasoning", "reviewing"],
    "additionalProperties": False
}

In [22]:
# 4. Retry wrapper
# =========================================================
@retry(
    retry=retry_if_exception_type((
        openai.RateLimitError,
        openai.APIConnectionError,
        openai.APITimeoutError,
        openai.InternalServerError
    )),
    wait=wait_exponential(multiplier=1, min=2, max=20),
    stop=stop_after_attempt(4),
    reraise=True
)
def get_answer(client, system_msg, user_msg, model="deepseek-chat"):
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    content = completion.choices[0].message.content
    return json.loads(content)

# =========================================================
# 5. Helper functions
# =========================================================
VALID_INDUSTRIES = {
    "Physical Production",
    "Infrastructure Systems",
    "Consumer Economy",
    "Financial Systems",
    "Healthcare Systems",
    "Information & Digital Services",
    "Media & Creative Industries",
    "Education & Research",
    "Public Sector & Civil Society",
    "Other"
}

def safe_str(x):
    if x is None:
        return ""
    return str(x)

def format_representative_docs(x, max_chars=12000):
    """
    Keep Representative_Docs as raw reference text, but cap length for cost control.
    """
    text = safe_str(x).strip()
    if not text:
        return "[No representative text provided]"
    return text[:max_chars]

def normalize_result(result):
    """
    Enforce schema-compatible output and strict fallback behavior.
    """
    fallback = {
        "industry": "Other",
        "relevant_keywords": [],
        "reasoning": "No obvious industry information.",
        "reviewing": "No clear topic in representative text or mostly boilerplate."
    }

    if not isinstance(result, dict):
        return fallback

    industry = result.get("industry")
    relevant_keywords = result.get("relevant_keywords", [])
    reasoning = result.get("reasoning", "")
    reviewing = result.get("reviewing", "")

    if industry not in VALID_INDUSTRIES:
        return fallback

    if not isinstance(relevant_keywords, list):
        relevant_keywords = []

    if not isinstance(reasoning, str) or not reasoning.strip():
        reasoning = fallback["reasoning"]

    if not isinstance(reviewing, str) or not reviewing.strip():
        reviewing = fallback["reviewing"]

    # strict fallback standard for Other
    if industry == "Other":
        normalized = {
            "industry": "Other",
            "relevant_keywords": [],
            "reasoning": reasoning.strip() if reasoning.strip() else fallback["reasoning"],
            "reviewing": reviewing.strip() if reviewing.strip() else fallback["reviewing"]
        }
    else:
        normalized = {
            "industry": industry,
            "relevant_keywords": [str(k) for k in relevant_keywords],
            "reasoning": reasoning.strip(),
            "reviewing": reviewing.strip()
        }

    try:
        validate(instance=normalized, schema=topic_classification_schema)
        return normalized
    except ValidationError:
        return fallback

In [23]:
# 6. Batch run
# =========================================================
answers = []

for idx, row in topic2_summary.iterrows():
    user_msg = topic_classification_user_template.format(
        dup_kw=row["dup_kw"],
        notdup_kw=row["notdup_kw"],
        Representative_Docs=format_representative_docs(row["Representative_Docs"], max_chars=12000)
    )

    try:
        raw_result = get_answer(
            client=client,
            system_msg=system_msg,
            user_msg=user_msg,
            model="deepseek-chat"
        )
        result = normalize_result(raw_result)
        result["api_status"] = "success"

    except Exception as e:
        err_msg = str(e)

        # stop immediately on insufficient balance
        if "Insufficient Balance" in err_msg or "402" in err_msg:
            print(f"Stopped at row {idx}: insufficient API balance.")
            break

        result = {
            "industry": None,
            "relevant_keywords": None,
            "reasoning": None,
            "reviewing": None,
            "api_status": f"failed: {err_msg}"
        }
        print(f"[Row {idx}] Failed: {e}")

    answers.append(result)

# =========================================================
# 7. Merge back
# =========================================================
answers_df = pd.DataFrame(answers)

topic2_summary_out = pd.concat(
    [topic2_summary.iloc[:len(answers)].reset_index(drop=True),
     answers_df.reset_index(drop=True)],
    axis=1
)

topic2_summary_out.head()

,Topic,Count,Rep_06,Rep_07,Rep_07_ad,Rep_uni,Share,Representative_Docs,dup_kw,notdup_kw,industry,relevant_keywords,reasoning,reviewing,api_status
0,-1,146540,"['digi', 'digi communications', 'fool', 'motle...","['digi', 'digi communications', 'share price',...","['digi', 'digi communications', 'prnewswire', ...","['nasdaq', 'digi', 'altman', 'llms', 'prnewswi...",0.541701,"[""BBAI Stock’s Price Performance & Valuation T...","[digi, digi communications, fool, motley, prne...","[communications announces, motley fool, forwar...",Financial Systems,"[digi, digi communications, traded, stocksbest...","Keywords like 'digi communications', 'traded',...",Financial investment analysis and stock recomm...,success
1,0,6987,"['drug discovery', 'breast', 'diseases', 'diag...","['clinical', 'imaging', 'drug discovery', 'dia...","['clinical', 'imaging', 'drug discovery', 'dia...","['clinical', 'disease', 'imaging', 'doctors', ...",0.025828,"[""Lunit to Power Germany's Largest Private Rad...","[drug discovery, breast, diseases, diagnostic,...","[physicians, oncology, patient care]",Healthcare Systems,"[drug discovery, diagnostic, diagnostics, clin...",The keywords strongly indicate healthcare doma...,Representative text discusses AI applications ...,success
2,1,3506,"['educators', 'classroom', 'cheating', 'assign...","['teachers', 'educators', 'classroom', 'cheati...","['teachers', 'educators', 'classroom', 'cheati...","['teachers', 'educators', 'classroom', 'teachi...",0.012960,['Schools prepare for Round 2 with AI | WYTV\n...,"[educators, classroom, cheating, assignments, ...",[classrooms],Education & Research,"[educators, classroom, faculty, higher educati...",The keywords strongly indicate an educational ...,Representative text discusses AI's impact on e...,success
3,2,3303,"['daily arabic', 'daily english', 'english dai...","['daily arabic', 'daily english', 'english dai...","['daily arabic', 'daily english', 'english dai...","['currencies', 'arabic', 'mena', 'menafn', 'in...",0.012210,['World and Middle East business and financial...,"[daily arabic, daily english, english daily, s...","[research weather, east business]",Financial Systems,"[stocks currencies, oil energy, east business,...",The keywords strongly indicate financial and e...,Representative text is primarily website navig...,success
4,3,2944,"['presale', 'mexc', 'solana', 'defi', 'price a...","['presale', 'xrp', 'decentralized', 'ethereum'...","['presale', 'ozak', 'xrp', 'ozak ai', 'decentr...","['presale', 'token', 'ozak', 'xrp', 'decentral...",0.010883,"['While XRP Holds $3, Ozak AI at $0.005 Offers...","[presale, mexc, solana, xrp, decentralized, et...","[defi, price analysis, staking, pepe, on-chain...",Financial Systems,"[presale, mexc, solana, xrp, decentralized, et...","Keywords like 'presale', 'tokens', 'coins', 's...",cryptocurrency investment analysis and presale...,success


In [24]:
DATA_DIR = "G:/我的云端硬盘/NLP_project"
topic2_summary_out.to_csv(f"{DATA_DIR}/topic2_summary_industry.csv", index=False)

---

### Post process

In [ ]:
DATA_DIR = "G:/我的云端硬盘/NLP_project"
df_blk_topics2 = pd.read_parquet(f"{DATA_DIR}/df_blk_topics2.parquet")
df_blk_topics2.head(2)

,doc_id,blk_id,url,date,title,block_text,block_char_len,topic,probability
0,0,0_0_0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",4123,0,1.0
1,0,0_0_1,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",Advertisement“It is allowing us to predict pro...,5170,0,1.0


In [30]:
# # check the rationale for the "other"
# topic2_summary_out[topic2_summary_out['industry'] == 'Other'][["Topic","Count",'dup_kw','Representative_Docs','industry','reasoning',"reviewing"]]

# Correct 2 industries that are wrongly classified as "other"
topic2_summary_out.loc[topic2_summary_out['Topic'] == 21, 'industry'] = "Information & Digital Services"
topic2_summary_out.loc[topic2_summary_out['Topic'] == 31, 'industry'] = "Public Sector & Civil Society"

In [31]:
topic2_summary_out['industry'].value_counts()

industry
Information & Digital Services    68
Financial Systems                 27
Media & Creative Industries       22
Other                             18
Physical Production               14
Public Sector & Civil Society      9
Infrastructure Systems             5
Education & Research               4
Consumer Economy                   3
Healthcare Systems                 2
Name: count, dtype: int64

In [45]:
valid_topic_map = (
    topic2_summary_out
    .loc[
        (topic2_summary_out["industry"] != "Other") &
        (topic2_summary_out["Topic"] != -1),
        ["Topic", "industry"]
    ]
    .copy()
)
display(valid_topic_map['industry'].value_counts())

valid_topic = set(valid_topic_map["Topic"])

df_blk_valid = df_blk_topics2[df_blk_topics2["topic"].isin(valid_topic)].copy()

df_blk_valid["topic"] = df_blk_valid["topic"].astype(int)
valid_topic_map["Topic"] = valid_topic_map["Topic"].astype(int)

topic_to_industry = dict(zip(valid_topic_map["Topic"], valid_topic_map["industry"]))
df_blk_valid["industry"] = df_blk_valid["topic"].map(topic_to_industry)

industry
Information & Digital Services    68
Financial Systems                 26
Media & Creative Industries       22
Physical Production               14
Public Sector & Civil Society      9
Infrastructure Systems             5
Education & Research               4
Consumer Economy                   3
Healthcare Systems                 2
Name: count, dtype: int64

In [ ]:
DATA_DIR = "G:/我的云端硬盘/NLP_project"
df_blk_valid.to_parquet(f"{DATA_DIR}/df_blk_valid_industry.parquet", index=False)

In [49]:
DATA_DIR = "G:/我的云端硬盘/NLP_project"
df_blk_valid = pd.read_parquet(f"{DATA_DIR}/df_blk_valid_industry.parquet")
df_blk_valid

,doc_id,blk_id,url,date,title,block_text,block_char_len,topic,probability,industry
0,0,0_0_0,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...","""A Gift To Humanity"": AlphaFold AI Predicts St...",4123,0,1.000000,Healthcare Systems
1,0,0_0_1,https://www.iflscience.com/a-gift-to-humanity-...,2022-07-28,"""A Gift To Humanity"": AlphaFold AI Predicts St...",Advertisement“It is allowing us to predict pro...,5170,0,1.000000,Healthcare Systems
2,4,0_0_0,https://thetaoofanarchy.substack.com/p/ai-feve...,2025-01-30,"""AI Fever!"" AI Is Animated Idiocy! Can This Hu...","""AI Fever!"" AI Is Animated Idiocy! Can This Hu...",2281,13,1.000000,Information & Digital Services
3,5,0_0_0,https://www.mondaq.com/patent/1581398/ai-will-...,2025-02-11,"""AI Will Not Replace Patent Attorneys—But Thos...","""AI Will Not Replace Patent Attorneys—But Thos...",5026,122,1.000000,Information & Digital Services
4,5,1_0_0,https://www.mondaq.com/patent/1581398/ai-will-...,2025-02-11,"""AI Will Not Replace Patent Attorneys—But Thos...",The Concept Of Stupidity (Part 1 Of 3): Know I...,1302,122,0.672588,Information & Digital Services
...,...,...,...,...,...,...,...,...,...,...
115573,148895,0_0_0,https://citylife.capetown/lt/uncategorized/big...,2023-12-20,„Mercedes“ CES parodoje pristato pažangiausią ...,„Mercedes“ CES parodoje pristato pažangiausią ...,2382,29,0.829604,Physical Production
115574,148897,0_0_0,https://ioplus.nl/en/posts/13m-grant-boosts-ai...,2025-12-17,€1.3M grant boosts AI hunt for early pancreati...,€1.3M grant boosts AI hunt for early pancreati...,2856,0,1.000000,Healthcare Systems
115575,148898,0_0_0,https://www.siliconrepublic.com/business/200bn...,2025-02-11,€200bn more mobilised for AI in Europe,€200bn more mobilised for AI in Europe\n\nThe ...,4982,106,0.645824,Public Sector & Civil Society
115576,148899,0_0_0,https://www.storyboard18.com/digital/%E2%82%B9...,2025-10-28,"₹12,000 cr on mute: OpenAI’s Generative music ...","₹12,000 cr on mute: OpenAI’s Generative music ...",1472,7,0.844302,Media & Creative Industries


### Spilt into smaller chunks

In [ ]:
df_blk_valid = (
    df_blk_valid
    .assign(block_id = df_blk_valid["doc_id"].astype(str) + "_" + df_blk_valid["blk_id"].astype(str))
    .drop(columns=["doc_id", "blk_id"])
)
df_blk_valid = df_blk_valid[["block_id"] + [c for c in df_blk_valid.columns if c != "block_id"]]

In [ ]:
import nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

def count_sentences(text):
    if not isinstance(text, str):
        return 0
    return len(sent_tokenize(text))

df_blk_valid["n_sent"] = df_blk_valid["block_text"].apply(count_sentences)

print(df_blk_valid["n_sent"].describe())

In [ ]:
def split_sentences(text):
    if not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    return sent_tokenize(text)

def make_sentence_chunks(
    sentences,
    chunk_size=12,
    overlap=2,
    min_tail_sent=4
):
    """
    Input:
        sentences: list[str]
    Output:
        list of dicts, each dict contains:
            - sent_start: inclusive
            - sent_end: exclusive
            - n_sent
            - chunk_text
    """
    n = len(sentences)
    if n == 0:
        return []
    
    if n <= chunk_size:
        return [{
            "sent_start": 0,
            "sent_end": n,
            "n_sent": n,
            "chunk_text": " ".join(sentences).strip()
        }]
    
    step = chunk_size - overlap
    chunks = []
    start = 0
    
    while start < n:
        end = min(start + chunk_size, n)
        
        # 如果这是最后一块，直接收掉
        if end == n:
            chunks.append({
                "sent_start": start,
                "sent_end": end,
                "n_sent": end - start,
                "chunk_text": " ".join(sentences[start:end]).strip()
            })
            break
        
        # 看一下剩余尾巴长度
        remaining = n - end
        
        # 如果尾巴太短，就不要再单独开新块了，
        # 直接把当前块扩到结尾
        if remaining < min_tail_sent:
            chunks.append({
                "sent_start": start,
                "sent_end": n,
                "n_sent": n - start,
                "chunk_text": " ".join(sentences[start:n]).strip()
            })
            break
        
        # 正常收当前块
        chunks.append({
            "sent_start": start,
            "sent_end": end,
            "n_sent": end - start,
            "chunk_text": " ".join(sentences[start:end]).strip()
        })
        
        start += step
    
    return chunks

def build_subchunks_df(
    df,
    text_col="block_text",
    block_id_col="block_id",
    chunk_size=12,
    overlap=2,
    min_tail_sent=4
):
    rows = []
    
    meta_cols = [c for c in [
        block_id_col, "url", "date", "title", "topic", "probability", "industry"
    ] if c in df.columns]
    
    for _, row in df.iterrows():
        text = row[text_col]
        sents = split_sentences(text)
        
        chunks = make_sentence_chunks(
            sents,
            chunk_size=chunk_size,
            overlap=overlap,
            min_tail_sent=min_tail_sent
        )
        
        for j, ch in enumerate(chunks):
            out = {col: row[col] for col in meta_cols}
            out["chunk_id"] = j
            out["chunk_uid"] = f"{row[block_id_col]}_{j}"
            out["sent_start"] = ch["sent_start"]
            out["sent_end"] = ch["sent_end"]
            out["n_sent"] = ch["n_sent"]
            out["chunk_text"] = ch["chunk_text"]
            rows.append(out)
    
    return pd.DataFrame(rows)

In [ ]:
df_subchunks = build_subchunks_df(
    df_blk_valid,
    text_col="block_text",
    block_id_col="block_id",
    chunk_size=12,
    overlap=2,
    min_tail_sent=4
)

df_subchunks['chunk_len'] = df_subchunks['chunk_text'].str.len()
df_subchunks = df_subchunks.drop_duplicates(subset=['chunk_text'], keep='first')

df_subchunks['chunk_len'].describe()


In [ ]:
df_subchunks.to_parquet(f"{DATA_DIR}/df_subchunks_raw.parquet", index=False)